In [2]:
import sys
sys.path.append('../../Classify_articles')

from file_management import get_files_dir,check_save_file
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()


import urllib.request, urllib.parse, urllib.error
import ssl
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

import re
import pickle
import pandas as pd

from functools import reduce

from extract_NCBI import *


# Scrap database to geth the files paths


Path were the htmls were downloaded: 'https://bitbucket.org/jdwinkler/laser_release/src/master/database_store/'

In [3]:
laser_path = INPUT_DIR+'/LASER/LASER_webfiles'

In [4]:
with open(laser_path+"/2014.html", "r", encoding='latin') as f:
    text= f.read()
with open(laser_path+"/2015.html", "r", encoding='latin') as f:
    text2= f.read()


In [5]:
papers_2014 = re.findall(r"Record.{1,30}\.txt", text)
papers_2014 = set(papers_2014)
print(len(papers_2014))

papers_2015 = re.findall(r"Record.{1,30}\.txt", text2)
papers_2015 = set(papers_2015)
print(len(papers_2015))


313
143


In [8]:
papers_2015

{'Record1423246634.26.txt',
 'Record1445326599.9.txt',
 'Record1445327759.27.txt',
 'Record1445330168.98.txt',
 'Record1445332284.92.txt',
 'Record1445407841.78.txt',
 'Record1445408957.51.txt',
 'Record1445409885.24.txt',
 'Record1445411109.2.txt',
 'Record1445493164.91.txt',
 'Record1445494730.2.txt',
 'Record1445496993.81.txt',
 'Record1445497569.24.txt',
 'Record1445579670.04.txt',
 'Record1445581702.36.txt',
 'Record1445582555.1.txt',
 'Record1445583566.62.txt',
 'Record1445843295.79.txt',
 'Record1445845113.42.txt',
 'Record1445847706.42.txt',
 'Record1445848934.32.txt',
 'Record1445924852.85.txt',
 'Record1445925852.59.txt',
 'Record1445926619.88.txt',
 'Record1445928204.93.txt',
 'Record1446014884.48.txt',
 'Record1446016635.48.txt',
 'Record1446017475.49.txt',
 'Record1446018258.3.txt',
 'Record1446177836.98.txt',
 'Record1446179017.02.txt',
 'Record1446183606.86.txt',
 'Record1446186204.92.txt',
 'Record1446187861.12.txt',
 'Record1446188933.12.txt',
 'Record1446189740.63.txt

In [9]:
papers_sets = {"2014":list(papers_2014), "2015":list(papers_2015)}


# Get the tags

## All of them 

In [10]:
list_tags = []
for year in papers_sets:
    
    papers = [title.split('.txt')[0].split('Record')[1] for title in papers_sets[year]]
    laser_url = "https://bitbucket.org/jdwinkler/laser_release/raw/f6ce080a8993ee259c4914ce92f83b1f966bab2d/database_store/"
    laser_url = laser_url+year+'/Record'
    
    papers_def = {key: {} for key in papers}
    failed = True
    
    for record in papers:
        failed = True
        url = laser_url+record+'.txt'
        
        while failed:
            try:
                html = urllib.request.urlopen(url, context=ctx)
                failed = False
            except:
                print('intento de nuevo ',html)
                
        for line in html:
                line=line.decode().rstrip()
                tag = line.split('=')[0].rstrip()
                if tag not in list_tags:
                    list_tags.append(tag)

In [18]:
full_tags = {'Counts':len(list_tags),
             'Full_list':list_tags,
             'Set':list(set(list_tags))}

In [20]:
#pd.DataFrame.from_dict(full_tags).to_json('All_LASER_TAGS.json')

## Mutants and mutations with levels

In [21]:
level_mut = {'lvl1':[],'lvl2':[],'lvl3':[],'lvl4':[]}

for tag in full_tags['Set']:
    
    if 'Mutant' in tag:
        muts = tag.split('.')
        level_mut['lvl1'].append(muts[0])
        
        if len(muts) > 1:
            level_mut['lvl2'].append(muts[1])
            
            if len(muts)>2:
                level_mut['lvl3'].append(muts[2])
                
                if len(muts)>3:
                    level_mut['lvl4'].append(muts[3])
                    
for n in level_mut:
    level_mut[n] = set(level_mut[n])

In [22]:
# pd.DataFrame.from_dict([level_mut])
# with open('mut_lvl_laser.pickle', 'wb') as handle:
#   pickle.dump(level_mut, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Extract information 

In [23]:
def create_primary_tags(papers, record):
    #print('***',record,'***')
    for tag in full_tags['Set']:
        if 'Mutant' not in tag:
            papers[record][tag]=None
    
    return(papers)
                              
def get_LASER_html(url, record):
        failed = True
        url = url+record+'.txt'
        while failed:
            try:
                html = urllib.request.urlopen(url, context=ctx)
                failed = False
            except:
                print('Try again ',url)
                
        return(html)
    
def extract_mutants(papers, tag, value, record):
    muts = tag.split('.')
    dic_mut = reduce(lambda res, cur: {cur: res}, reversed(muts), value)
    
    if len(muts) > 1:
        papers[record][muts[0]] =  papers[record].get(muts[0],{})
        if len(muts) == 2:
            papers[record][muts[0]][muts[1]]= value
            
        elif len(muts)>2:
            papers[record][muts[0]][muts[1]] =  papers[record][muts[0]].get(muts[1],{})
            if len(muts) ==3:
                if ',' not in value and [muts[2]] == 'GeneMutation':
                    papers[record][muts[0]][muts[1]][muts[2]]= {'value':value}
                else:
                    papers[record][muts[0]][muts[1]][muts[2]]={}
                    for mut in value.split(','):
                        papers[record][muts[0]][muts[1]][muts[2]][mut.strip('"')]={}
                        
            elif len(muts)>3:
                papers[record][muts[0]][muts[1]][muts[2]] =  papers[record][muts[0]][muts[1]].get(muts[2],{})
                if len(muts) == 4:
                    #print( papers[record][muts[0]][muts[1]][muts[2]],muts )

                    papers[record][muts[0]][muts[1]][muts[2]][muts[3]]=value
                else:
                    print('FAIL')
                    
    return(papers)
    
def read_LASER_html(html, papers, record):
    numut = None
    
    for line in html:
        line = line.decode().rstrip()
        line = line.split('=')
        
        tag = line[0].strip()
        value = line[1].strip()

        if 'NumberofMutants' in tag:
            numut = int(value.strip('"'))
            for mutant in range(numut):
                papers[record]['Mutant'+str(mutant+1)]={}
                
        elif not(numut) and '_Gonzalez_BetaOxidation' == record:
            numut = 4
            for mutant in range(numut):
                papers[record]['Mutant'+str(mutant+1)]={}
                
        elif numut and 'Mutant' in tag:
            #print(line)
            papers = extract_mutants(papers, tag, value, record)
            
        else:
            papers[record][tag] = value
                
    return(papers)

In [24]:
parsed_data = {}
for year in papers_sets:

    papers = [title.split('.txt')[0].split('Record')[1] for title in papers_sets[year]]
    laser_url = "https://bitbucket.org/jdwinkler/laser_release/raw/f6ce080a8993ee259c4914ce92f83b1f966bab2d/database_store/"
    laser_url = laser_url+year+'/Record'
    
    papers_def = {key: {} for key in papers}
    
    for record in papers_def.keys():
        papers_def = create_primary_tags(papers_def, record)
        html = get_LASER_html(laser_url, record)
        papers_def = read_LASER_html(html, papers_def, record)
        
    
    parsed_data[year]=pd.DataFrame(papers_def)
                    

In [25]:
datos = pd.DataFrame([parsed_data])

# Make a database using this information 

In [26]:
check_save_file(datos,'parsed_LASER_DB.json','LASER',input_dir=True)

Saved file in: /Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Input/LASER/parsed_LASER_DBnew_v.json


# Get some articles

In [59]:
DOIs = pd.concat([datos.loc[:,'2014'][0],datos.loc[:,'2015'][0]],axis=1).T.DOI

In [76]:
Titles = pd.concat([datos.loc[:,'2014'][0],datos.loc[:,'2015'][0]],axis=1).T.Title

In [77]:
zip(DOIs,Titles)

In [113]:
fix_doi = [n.split('doi:')[-1] for n in DOIs]
fix_doi = [n.split('doi/')[-1] for n in fix_doi]
fix_doi = [n.split('doi.org/')[-1] for n in fix_doi]


In [114]:
change = {
    '"10.1128/AEM.71.12.7880/7887.2005"': '"10.1128/AEM.71.12.7880-7887.2005"',
    '"10.1016/S1096-7176(03)00046-6"': '"10.1016/s1096-7176(03)00046-6"',
    '"10.1128/AEM.67.1.148154.2001"': '"10.1128/aem.67.1.1-5.2001"',
    '"10.1128/AEM.69.10.5772/5781.2003"': '"10.1128/AEM.69.10.5772-5781.2003"',
    '"0273-2289/06/129-132/795-807/$30.00"': '"10.1385/abab:131:1:795"',
    '"0273-2289/98/70-72--0919510.50"': '"10.1007/BF02920202"',
    '"0099-2240/99/$04.0010"':'"10.1128/AEM.65.4.1384-1389.1999"',
    '"0099-2240/00/$04.0010"':'"10.1128/aem.66.12.5383-5386.2000"',
    '"0099-2240/87/102420-06$02.00/0"': '"10.1128/aem.53.10.2420-2425.1987"',
    '"0099-2240/97/$04.0010"':'"10.1128/aem.63.6.2442-2445.1997"',
    '"8756-7938/95/3011-0475$09.0"':'"10.1021/bp00034a019"',
    '"0273-2289/98/70-72-0187$11.00-X"':'"10.1007/BF02920135 "',
    '"APPLIED AND ENVIRONMENTAL MICROBIOLOGY, May 1996, p. 1808-1810"':'"10.1128/aem.62.5.1808-1810.1996"',
    '"APPLIED AND ENVIRONMENTAL MICROBIOLOGY, June 1983, p. 1838-1847"':'"10.1128/aem.45.6.1838-1847.1983"',
    '"J. Biochem. 123, 1088-1096 (1998)"':'"10.1093/oxfordjournals.jbchem.a022047"',
    '"0099-2240/94/$04.00+0"':'"10.1128/aem.60.11.3903-3908.1994"',
    '"Biotechnology Letters 21: 791795, 1999"' :'"10.1023/A:1005547827380"',
    '"Applied Biochemistry and Biotechnology, Vol. 34/35, 1992, Page 149"':'"10.1007/BF02920542"',
    '"Patrik R Jones"':'"10.1186/s13068-015-0231-1"'
}

In [115]:
updated_list = [change.get(item, item) for item in fix_doi]

for n, title in zip(updated_list,Titles):
    if not n.startswith('"10.'):
        print(n,title)
        print()

10.1038/nbt.2149" "Design of a dynamic sensor-regulator system for production of chemicals and fuels derived from fatty acids"

10.1186/1475-2859-11-93" "An inverse metabolic engineering approach for the design of an improved host platform for over-expression of recombinant proteins in Escherichia coli"

10.1038/nbt.1789" "Conversion of proteins into biofuels by engineering nitrogen flux"

10.1016/j.ymben.2012.08.008" "An integrated computational and experimental study for overproducing fatty acids in Escherichia coli"

10.1002/bit.22660" "A Process for Microbial Hydrocarbon Synthesis: Overproduction of Fatty Acids in Escherichia coli and Catalytic Conversion to Alkanes"

10.1073/pnas.1110740109" "Production of amorphadiene in yeast, and its conversion to dihydroartemisinic acid, precursor to the antimalarial agent artemisinin"



In [249]:
updated_dois = pd.DataFrame({'DOI':updated_list,'Title':Titles})

In [250]:
updated_dois = updated_dois.drop_duplicates().copy()

In [410]:
for n, title in zip(updated_dois.DOI,updated_dois.Title):
    if n in updated_dois.DOI[updated_dois.DOI.duplicated()].to_list():
        print(n,title)
        print()

In [253]:
updated_dois.loc[updated_dois.Title == '"Metabolic Engineering of a 1,2-Propanediol Pathway in Escherichia coli"','DOI'] = "10.1128/AEM.65.3.1180-1185.1999"

In [255]:
articles_ids = []
for n in updated_dois.DOI.str.strip('"').values:
    if '/' in n:
        article_id = search_entrez(n)
        if article_id['IdList']:
            articles_ids.append(article_id['IdList'][0])

/Users/elisamarquez/opt/anaconda3/lib/python3.9/site-packages/Bio/Entrez/__init__.py:694: UserWarning: 
            Email address is not specified.

            To make use of NCBI's E-utilities, NCBI requires you to specify your
            email address with each request.  As an example, if your email address
            is A.N.Other@example.com, you can specify it as follows:
               from Bio import Entrez
               Entrez.email = 'A.N.Other@example.com'
            In case of excessive usage of the E-utilities, NCBI will attempt to contact
            a user at the email address provided before blocking access to the
            E-utilities.
  warnings.warn(


In [256]:
file = INPUT_DIR+"/LASER/LASER_articles.csv"
collect_relevant_info(articles_ids, file, batches_len=200)

Failed to extract relevant info:  36883770
Information saved in /Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Input/LASER/LASER_articles.csv


In [411]:
print(len(updated_dois.DOI.str.strip('"').values),
     len(updated_dois.Title.str.strip('"')),
     len(articles_ids))

451 451 451


In [419]:
len(list(set(articles_ids)))

445

In [274]:
expected = zip(updated_dois.DOI.str.strip('"').values,updated_dois.Title.str.strip('"'),articles_ids)

In [538]:

# Load the LASER articles CSV
laser_articles = pd.read_csv('../../../files/Input/LASER/LASER_articles.csv', sep="\t")


In [539]:
laser_articles = laser_articles.drop_duplicates().copy()

In [540]:
laser_articles.shape

(444, 9)

In [541]:
expected

In [542]:
for DOI, Title, ID in zip(updated_dois.DOI.str.strip('"').values,updated_dois.Title.str.strip('"'),articles_ids):
    if int(ID) in laser_articles.PM_ID.to_list():
        laser_info = laser_articles.loc[laser_articles.PM_ID==int(ID)]
        laser_title = laser_info.Title.iloc[0]
        laser_DOI = laser_info.DOI.iloc[0]
        if laser_DOI.lower().strip('+')!=DOI.lower().split(' ')[0]:
            #print(laser_info)
            print(int(ID))
            print(laser_DOI, DOI,'doi')
            print()
            print(laser_title, Title)

            if DOI.lower().split(' ')[0] not in laser_articles.DOI.str.lower().str.strip('+').str.strip().to_list():
                print(DOI,'MISSING')
            print()
            print()

35642214
10.2147/JIR.S350109 10.1016/j.ijhydene.2014.06.054 doi

Microglia-Mediated Neuroinflammation: A Potential Target for the Treatment of Cardiovascular Diseases. Metabolic engineering of Escherichia coli strains for co-production of hydrogen and ethanol from glucose
10.1016/j.ijhydene.2014.06.054 MISSING


35642214
10.2147/JIR.S350109 10.1016/j.bej.2012.06.006 doi

Microglia-Mediated Neuroinflammation: A Potential Target for the Treatment of Cardiovascular Diseases. Production of pyruvate in Saccharomyces cerevisiae through adaptive evolution and rational cofactor metabolic engineering
10.1016/j.bej.2012.06.006 MISSING


35642214
10.2147/JIR.S350109 10.1016/j.ymben.2014.09.006i doi

Microglia-Mediated Neuroinflammation: A Potential Target for the Treatment of Cardiovascular Diseases. Engineering modular ester fermentative pathways in Escherichia coli
10.1016/j.ymben.2014.09.006i MISSING


34149274
10.1134/S1068162021030055 10.1134/S000368381008003X doi

Molecular Beacon DNA Probe

In [543]:
doi_dict = {"10.1016/j.bej.2012.06.006":"Pyruvate-decarboxylase (Pdc)-negative Saccharomyces cerevisiae has been proven as a suitable metabolic engineering platform to produce organic acids. S. cerevisiae BY5419 Pdc− strain cannot grow in batch cultures on synthetic medium with glucose as the sole carbon source, yet grows well on synthetic medium with ethanol or acetate. In this study, by combining adaptive evolution and cofactor engineering, we obtained a series of engineered yeasts that can produce pyruvate using glucose as sole carbon source. Differential expression of noxE, encoding a water-forming NADH oxidase from Lactococcus lactis, and udhA, encoding a soluble pyridine nucleotide transhydrogenase from Escherichia coli, was investigated. Of all the constructed recombinant strains, G2U1-A0 was able to produce 75.1 g l−1 pyruvate, increased 21% compared with the original strain A0. The production yield of this strain reached 0.63 g of pyruvate g of glucose−1. This study demonstrates that the fine regulation of intracellular NADH/NAD+ ratio is critical for cell metabolism and pyruvate production. Combining the adaptive evolution and fine regulation of intracellular NADH/NAD+ ratio provides a new strategy for improving the Pdc− strain engineering platform.",
"10.1134/S000368381008003X":"Multistage construction of an E. coli strain containing no foreign genes which is capable of producing butyrate has been carried out. At the first stage, deletions in gene fadR encoding a protein repressor of an operon for fatty acid degradation and gene aceF responsible for the synthesis of pyruvate dehydrogenase were introduced in the strain MG1655 genome. Then, a mutant obtained from the above strain by induced mutagenesis and capable of growth on ethanol as a sole carbon source under aerobic conditions was selected. It was shown that growth of the mutant on ethanol is provided by two mutations. One of them (a substitution: 257G → A) is located in the regulatory region of gene adhE that controls the synthesis of alcohol-dehydrogenase; the other, containing a substitution Glu568 → Lys, affects the structural portion of the gene. As a result of the consequent mutagenesis of the obtained strain and selection on indicating media, variants capable of growing on butyrate and butanol as sole carbon sources and putatively bearing mutations in gene atoC (encoding transcriptional activator of atoDAB operon) were selected. At the last stage of the work, gene atoB, encoding the synthesis of the thiolase II enzyme, was placed under the control of a constitutive promoter P tet , and the functional allele of gene aceF was introduced. The resulting E. coli strain (ΔfadR, adhE, atoC, P tet -atoB) accumulates 800 mg/l of butyrate upon growth on glucose-containing medium under semi-anaerobic (oxygen limited) conditions. Introduction of an additional deletion in gene ldhA encoding lactate dehydrogenase in the strain genome leads to a further growth of a butyrate production up to 1.3 g/l.",
"10.1023/A:1005547827380":"Metabolic engineering of the early non-mevalonate terpenoid pathway of Escherichia coli was carried out to increase the supply of prenyl pyrophosphates as precursor for carotenoid production. Transformation with the genes dxs for over-expression of 1-deoxy-d-xylulose 5-phosphate synthase, dxr for 1-deoxy-d-xylulose 5-phosphate reductoisomerase and idi encoding an isopentenyl pyrophosphate stimulated carotenogenesis up to 3.5-fold. Co-transformation of idi with either dxs or dxr had an additive effect on ß-carotene and zeaxanthin production which reached 1.6 mg g−1 dry wt.",
 "10.1016/j.ijhydene.2014.06.054":"The co-production of H2 and ethanol from glucose was studied to address the low H2 production yield in dark fermentation. Several mutant strains devoid of ackA-pta, pfkA or pgi were developed using Escherichia coli BW25113 ΔhycA ΔhyaAB ΔhybBC ΔldhA ΔfrdAB as base strain. Disruption of ackA-pta eliminated acetate production during glucose fermentation but resulted in the secretion of a significant amount of pyruvate (0.73 mol mol−1 glucose) without improving the co-production of H2 and ethanol. When pfkA or pgi was further disrupted to enhance NAD(P)H supply by diverting the carbon flux from Embden-Meyerhof-Parnas (EMP) pathway to the pentose phosphate pathway (PPP), the cell growth of both strains was severely impaired under anaerobic conditions, and only the ΔpfkA mutant could recover its growth after adaptive evolution. The production yields of the ΔpfkA strain (H2, 1.03 mol mol−1 glucose and ethanol, 1.04 mol mol−1 glucose) were higher than those of the pfkA+ strain (H2, 0.69 mol mol−1 glucose and ethanol, 0.95 mol mol−1 glucose), but pyruvate excretion was not reduced. The excessive excretion of pyruvate in the ΔpfkA mutant was attributed to an insufficient NAD(P)H supply caused by the diversion of carbon flux from the EMP pathway to the Entner-Doudoroff pathway (EDP), rather than the PPP as intended. This study suggests that co-production of H2 and ethanol from glucose is possible, but further metabolic pathway engineering is required to fully activate PPP under anaerobic conditions.",
    "10.1016/j.procbio.2013.10.002":"The compound 1,2,4-butanetriol (BT) is a valuable chemical used in the production of plasticizers, polymers, cationic lipids and other medical applications, and is conventionally produced via hydrogenation of malate. In this report, BT is biosynthesized by an engineered Escherichia coli from d-xylose. The pathway: d-xylose → d-xylonate → 2-keto-3-deoxy-d-xylonate → 3,4-dihydroxybutanal → BT, was constructed in E. coli by recruiting a xylose dehydrogenase and a keto acid decarboxylase from Caulobacter crescentus and Pseudomonas putida, respectively. Authentic BT was detected from cultures of the engineered strain. Further improvement on the strain was performed by blocking the native d-xylose and d-xylonate metabolic pathways which involves disruption of xylAB, yjhH and yagE genes in the host chromosome. The final construct produced 0.88 g L−1 BT from 10 g L−1 d-xylose with a molar yield of 12.82%. By far, this is the first report on the direct production of BT from d-xylose by a single microbial host. This may serve as a starting point for further metabolic engineering works to increase the titer of BT toward industrial scale viability.",
    "10.1016/j.ymben.2014.09.006i":"Sensation profiles are observed all around us and are made up of many different molecules, such as esters. These profiles can be mimicked in everyday items for their uses in foods, beverages, cosmetics, perfumes, solvents, and biofuels. Here, we developed a systematic ‘natural’ way to derive these products via fermentative biosynthesis. Each ester fermentative pathway was designed as an exchangeable ester production module for generating two precursors− alcohols and acyl-CoAs that were condensed by an alcohol acyltransferase to produce a combinatorial library of unique esters. As a proof-of-principle, we coupled these ester modules with an engineered, modular, Escherichia coli chassis in a plug-and-play fashion to create microbial cell factories for enhanced anaerobic production of a butyrate ester library. We demonstrated tight coupling between the modular chassis and ester modules for enhanced product biosynthesis, an engineered phenotype useful for directed metabolic pathway evolution. Compared to the wildtype, the engineered cell factories yielded up to 48 fold increase in butyrate ester production from glucose.",
    "10.1016/j.enzmictec.2009.08.003":"Succinate fermentation was investigated in Escherichia coli strains overexpressing cyanobacterium Anabaena sp. 7120 ecaA gene encoding carbonic anhydrase (CA). In strain BL21 (DE3) bearing ecaA, the activity of CA was 21.8 U mg−1 protein, whereas non-detectable CA activity was observed in the control strain. Meanwhile, the activity of phosphoenolpyruvate carboxylase (PEPC) increased from 0.2 U mg−1 protein to 1.13 U mg−1 protein. The recombinant bearing ecaA reached a succinate yield of 0.39 mol mol−1 glucose at the end of the fermentation. It was 2.1-fold higher than that of control strain which was just 0.19 mol mol−1 glucose. EcaA gene was also introduced into E. coli DC1515, which was deficient in glucose phosphotransferase, lactate dehydrogenase and pyruvate:formate lyase. Succinate yield can be further increased to 1.26 mol mol−1 glucose. It could be concluded that the enhancement of the supply of HCO3− in vivo by ecaA overexpression is an effective strategy for the improvement of succinate production in E. coli.",
    "10.1016/j.ymben.2015.03.015i":"Poly(3-hydroxypropionate) (P3HP) is the strongest family member of microbial polyhydroxyalkanoates (PHA) synthesized by bacteria grown on 1,3-propandiol or glycerol. In this study synthesis pathways of P3HP and its copolymer P3HB3HP of 3-hydroxybutyrate (3HB) and 3-hydroxypropionate (3HP) were assembled respectively to allow their synthesis from glucose, a more abundant carbon source. Recombinant Escherichia coli was constructed harboring the P3HP synthetic pathway consisting of heterologous genes encoding glycerol-3-phosphate dehydrogenase (gpd1), glycerol-3-P phosphatase (gpp2) from Saccharomyces cerevisiae that catalyzes formation of glycerol from glucose, and genes coding glycerol dehydratase (dhaB123) with its reactivating factors (gdrAB) from Klebsiella pneumoniae that transfer glycerol to 3-hydroxypropionaldehyde, as well as gene encoding propionaldehyde dehydrogenase (pdup) from Salmonella typhimurium which converts 3-hydroxypropionaldehyde to 3-hydroxypropionyl-CoA, together with the gene of PHA synthase (phaC) from Ralstonia eutropha which polymerizes 3-hydroxypropionyl-CoA into P3HP. When phaA and phaB from Ralstonia eutropha respectively encoding β-ketothiolase and acetoacetate reductase, were introduced into the above P3HP producing recombinant E. coli, copolymers poly(3-hydroxybutyrate-co-3-hydroxypropionate) (P3HB3HP) were synthesized from glucose as a sole carbon source. The above E. coli recombinants grown on glucose LB medium successfully produced 5 g/L cell dry weight containing 18% P3HP and 42% P(3HB-co-84 mol% 3HP), respectively, in 48 h shake flask studies."
           }


In [544]:
updated_dois.loc[updated_dois.DOI.str.strip('"').isin(corrections.DOI.to_list())]

,DOI,Title
1419968290.81,"""10.1016/j.ijhydene.2014.06.054""","""Metabolic engineering of Escherichia coli str..."
1422031154.03,"""10.1016/j.bej.2012.06.006""","""Production of pyruvate in Saccharomyces cerev..."
1419700122.69,"""10.1016/j.ymben.2014.09.006i""","""Engineering modular ester fermentative pathwa..."
1419614251.13,"""10.1134/S000368381008003X""","""Construction of a Butyrate-Producing E. coli ..."
1447657780.0,"""10.1023/A:1005547827380""","""Metabolic engineering of the terpenoid biosyn..."
1446532566.22,"""10.1016/j.enzmictec.2009.08.003""","""Improvement of succinate production by overex..."
1450238255.03,"""10.1016/j.procbio.2013.10.002""","""Direct bioconversion of d-xylose to 1,2,4-but..."
1449201641.39,"""10.1016/j.ymben.2015.03.015i""","""Production of poly(3-hydroxypropionate) and p..."


In [545]:
corrections= pd.DataFrame.from_dict([doi_dict]).T.reset_index()
corrections.columns = ['DOI','Abstract']

In [546]:
mini = updated_dois.loc[updated_dois.DOI.str.strip('"').isin(corrections.DOI.to_list())].copy()
mini.DOI = mini.DOI.str.strip('"')
mini.Title = mini.Title.str.strip('"')

In [547]:
add_info = pd.concat([mini.set_index('DOI'),corrections.set_index('DOI')],axis=1).reset_index()

In [548]:
add_info.loc[:,'PM_ID'] = None
add_info.loc[:,'PMC_ID'] = None
add_info.loc[:,'Type'] = 'Journal Article'
add_info.loc[:,'Author'] = None

In [549]:
add_info.loc[:,'Journal'] = [
    "International Journal of Hydrogen Energy",
    "Biochemical Engineering Journal",
    "Metabolic Engineering",
    "Applied Biochemistry and Microbiology",
    "Biotechnology Letters",
    "Enzyme and Microbial Technology",
    "Process Biochemistry",
    "Metabolic Engineering"
]

In [550]:
add_info.loc[:,'Year'] = [
    2014,  # DOI: 10.1016/j.ijhydene.2014.06.054
    2012,  # DOI: 10.1016/j.bej.2012.06.006
    2014,  # DOI: 10.1016/j.ymben.2014.09.006i
    2010,  # DOI: 10.1134/S000368381008003X
    1999,  # DOI: 10.1023/A:1005547827380
    2009,  # DOI: 10.1016/j.enzmictec.2009.08.003
    2013,  # DOI: 10.1016/j.procbio.2013.10.002
    2015   # DOI: 10.1016/j.ymben.2015.03.015i
]

In [551]:
updates = {}
remove_ID = []
for DOI, Title, ID in zip(updated_dois.DOI.str.strip('"').values, updated_dois.Title.str.strip('"'), articles_ids):
    if int(ID) in laser_articles.PM_ID.to_list():
        laser_info = laser_articles.loc[laser_articles.PM_ID == int(ID)]
        laser_DOI_cleaned = laser_info.DOI.iloc[0].lower().strip().strip('+').strip()
        DOI_cleaned = DOI.lower().strip().strip('+').split(' ')[0]
        if laser_DOI_cleaned != DOI_cleaned:
            if DOI_cleaned in doi_dict:
                updates[DOI_cleaned] = {
                    'Title': Title.strip('"'),
                    'Abstract': doi_dict[DOI],
                    'DOI': DOI_cleaned,
                    'PM_ID':None
                }
            remove_ID.append(int(ID))


In [552]:
laser_articles = laser_articles.loc[~laser_articles.PM_ID.isin( remove_ID)].copy()

In [558]:
curated_laser = pd.concat([laser_articles,add_info]).reset_index()

/var/folders/v6/y2v6wwn93yj5vvhl558fx2zc0000gn/T/ipykernel_41600/2284286147.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  curated_laser = pd.concat([laser_articles,add_info]).reset_index()


In [563]:

curated_laser = curated_laser.drop(columns=['index'])

In [568]:
curated_laser.Journal.value_counts()

Journal
Metabolic engineering                                                              129
Applied and environmental microbiology                                              63
Biotechnology and bioengineering                                                    49
Microbial cell factories                                                            44
Applied microbiology and biotechnology                                              26
Proceedings of the National Academy of Sciences of the United States of America     12
ACS synthetic biology                                                               11
Nature biotechnology                                                                11
Journal of industrial microbiology & biotechnology                                  10
Journal of biotechnology                                                             7
Biotechnology for biofuels                                                           7
Biotechnology journal              

In [564]:
file = '/Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Input/LASER/LASER_articles_curated.csv'

In [566]:
curated_laser.to_csv(file,sep='\t',index=False)